<a href="https://colab.research.google.com/github/motahbashir/Turkish-ASR-/blob/main/Testing_whisper_on_merged_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets soundfile librosa pandas -q

In [ ]:
from datasets import load_dataset

First Dataset

In [ ]:
dataset1=load_dataset("ysdede/commonvoice_17_tr_fixed",
                     streaming=True,
                     split='train')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.81k [00:00<?, ?B/s]

In [ ]:
from datasets.features import Audio
tr_test1_16k= dataset1.cast_column("audio", Audio(sampling_rate=16000)
                                 )

In [ ]:
sampled_dataset1 = dataset1.take(2000)

In [ ]:
list_commonvoice = []
for sample in sampled_dataset1:
  list_commonvoice.append({
      "id": sample.get("path", "unknown_id"),
        "audio_array": sample["audio"]["array"],
        "sampling_rate": sample["audio"]["sampling_rate"],
        "text": sample["transcription"]
    }
  )

Seccond Dataset

In [ ]:
dataset2=load_dataset("FBK-MT/Speech-MASSIVE",
                     'tr-TR',
                     streaming=True,
                     split='validation')

README.md:   0%|          | 0.00/65.6k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
tr_test2_16k=dataset2.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
sampled_dataset2 =dataset2.take(1000)

In [ ]:
list_Massive = []
for sample in sampled_dataset2:
  list_Massive.append({
      "id": sample.get("path", "unknown_id"),
        "audio_array": sample["audio"]["array"],
        "sampling_rate": sample["audio"]["sampling_rate"],
        "text": sample["utt"]
    }
  )

In [ ]:
print(list_Massive[0]["text"])

sosyal bilgi


Third Dataset

In [ ]:
dataset3=load_dataset("ysdede/khanacademy-turkish",
                      streaming=True,
                      split='test')

README.md:   0%|          | 0.00/116k [00:00<?, ?B/s]

In [ ]:
tr_test3_16k= dataset3.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
sampled_dataset3 =dataset3.take(1000)

In [ ]:
sampled_dataset3 = dataset3.take(1000)
list_KhanAcademy = []
for sample in sampled_dataset3:
  list_KhanAcademy.append({
      "id": sample.get("path", "unknown_id"),
        "audio_array": sample["audio"]["array"],
        "sampling_rate": sample["audio"]["sampling_rate"],
        "text": sample["transcription"]
    }
  )

In [ ]:
print(list_KhanAcademy[0]["text"])

Yerin yakınında yer çekimi ivmesi 9,8 metre bölü saniye kare ve aşağı yönlüdür.


Final Merged List

In [ ]:
final_merged_list=list_commonvoice+list_Massive+list_KhanAcademy

In [ ]:
import random
random.shuffle(final_merged_list)

Testing Whiser Model

In [ ]:
!pip install transformers accelerate librosa jiwer pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from tqdm import tqdm
df_all = pd.DataFrame(final_merged_list)
df_test = df_all.head(1000).copy()


In [ ]:
!pip install faster-whisper -q
import torch
from faster_whisper import WhisperModel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 54.0 MB/s eta 0:00:00


In [ ]:
import torch
from faster_whisper import WhisperModel
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "float32"

print(f"loading Whisper Large-v3 on : {device.upper()}")
model = WhisperModel("large-v3", device=device, compute_type=compute_type)

df_test['pred_whisper_raw'] = ""

for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    audio_array = row['audio_array']
    segments, info = model.transcribe(audio_array, language="tr", beam_size=5, temperature=0.0)

    chunk_text = ""
    for segment in segments:
        chunk_text += segment.text + " "

    df_test.at[index, 'pred_whisper_raw'] = chunk_text.strip()

    if index % 50 == 0:
        df_test.to_csv("whisper_backup_1000.csv", index=False)



loading Whisper Large-v3 on : CUDA


100%|██████████| 1000/1000 [13:14<00:00,  1.26it/s]


In [ ]:
import re
import jiwer


def clean_turkish_text(text):
    if not isinstance(text, str):
        return ""
    turkish_map = {"I": "ı", "İ": "i", "Ş": "ş", "Ğ": "ğ", "Ç": "ç", "Ö": "ö", "Ü": "ü"}
    for old, new in turkish_map.items():
        text = text.replace(old, new)
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df_test['clean_ground_truth'] = df_test['text'].apply(clean_turkish_text)
df_test['clean_pred_whisper'] = df_test['pred_whisper_raw'].apply(clean_turkish_text)

for idx, row in df_test.head(10).iterrows():
    print(f"[{idx + 1}] REF : {row['clean_ground_truth']}")
    print(f"    PRED: {row['clean_pred_whisper']}")
    print("-" * 50)


references = df_test['clean_ground_truth'].tolist()
hypotheses = df_test['clean_pred_whisper'].tolist()

total_wer = jiwer.wer(references, hypotheses)
total_cer = jiwer.cer(references, hypotheses)

print(f" (Total WER): {round(total_wer * 100, 2)}%")
print(f" (Total CER): {round(total_cer * 100, 2)}%")


df_test.to_csv("final_results_1000_chunks.csv", index=False)

[1] REF : başımı yavaşça kaldırdım
    PRED: başımı yavaşça kaldırdım
--------------------------------------------------
[2] REF : odanın tenhalaştığı bir anda arkadaşım başını kaldırarak
    PRED: odanın tenhalaştığı bir anda arkadaşın başını kaldırarak
--------------------------------------------------
[3] REF : kaymakam ince parmakların tombul yanağında bıraktığı izleri ovuşturarak masanın yanına sıçradı
    PRED: kaymakam ince parmakların topul yananda bıraktığı izleri ovuşturarak masanın yanına sıçradı
--------------------------------------------------
[4] REF : bu da ne demek şimdi
    PRED: bu da ne demek şimdi
--------------------------------------------------
[5] REF : ankara valisi kim
    PRED: ankara valisi kim
--------------------------------------------------
[6] REF : ocağın içi hayret edilecek kadar beyazdı
    PRED: ocağın içi hayret edilecek kadar beyazdı
--------------------------------------------------
[7] REF : yarın sabah için hiç alarmım var mı
    PRED: yarın s